In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')


/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd
import os

In [4]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sub   = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

Train: (2000, 8), Test: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
most_common = train['answer'].value_counts().idxmax()
print(f"Most common answer: {most_common}")

sub['answer'] = most_common
print(sub.head())

Most common answer: B
   ID Prediction answer
0   1      A B C      B
1   2      A B C      B
2   3      A B C      B
3   4      A B C      B
4   5      A B C      B


In [8]:
# Load fresh sample submission
sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

print("Sample submission columns:", sub.columns.tolist())
print(sub.head())

Sample submission columns: ['ID', 'Prediction']
   ID Prediction
0   1      A B C
1   2      A B C
2   3      A B C
3   4      A B C
4   5      A B C


In [9]:
# Use the correct column name from sample_submission
sub['Prediction'] = most_common   # replace 'Prediction' with whatever column name printed above

sub.to_csv('submission.csv', index=False)
print("Done!")
print(sub.head())

Done!
   ID Prediction
0   1          B
1   2          B
2   3          B
3   4          B
4   5          B


# Milestone 1:-

In [1]:
import pandas as pd
import numpy as np
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

### Question 1: Answer Distribution
Calculate the frequency distribution of the correct answer (A, B, C, D, E) in `train.csv`. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [2]:
counts = df['answer'].value_counts()
sum_most_least = counts.max() + counts.min()
sum_most_least

814

### Question 2: Vocabulary Size

In [3]:
def clean_text(text):
    text = text.lower()
    return text.translate(str.maketrans('', '', string.punctuation))

cleaned_prompts = df['prompt'].apply(clean_text)
vocab = set(' '.join(cleaned_prompts).split())
len(vocab)

859

### Question 3: Stop Word Filtering

In [4]:
row1_tokens = clean_text(df.iloc[0]['prompt']).split()
filtered_tokens = [w for w in row1_tokens if w not in ENGLISH_STOP_WORDS]
len(filtered_tokens)

13

### Question 4: TF-IDF Vocabulary

In [6]:
combined_text = df['prompt'] + " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(combined_text)
len(vectorizer.get_feature_names_out())

2762

### Question 5: Cosine Similarity

In [7]:
prompt_vec = vectorizer.transform([df.iloc[0]['prompt']])
option_a_vec = vectorizer.transform([df.iloc[0]['A']])
round(cosine_similarity(prompt_vec, option_a_vec)[0][0], 4)

np.float64(0.272)

### Question 6: Similarity Matching Accuracy

In [8]:
def get_best_option(row):
    options = ['A', 'B', 'C', 'D', 'E']
    prompt_v = vectorizer.transform([row['prompt']])
    scores = {opt: cosine_similarity(prompt_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return max(scores, key=scores.get)

matches = df.apply(lambda row: get_best_option(row) == row['answer'], axis=1)
matches.mean() * 100

np.float64(13.55)

### Question 7 & 8: MAP@3 Scoring Functions

In [9]:
def map_at_3(truth, predictions):
    if truth in predictions[:3]:
        rank = predictions.index(truth) + 1
        return 1 / rank
    return 0

### Question 9: Majority Class Baseline

In [ ]:
top_answers = counts.index.tolist()[:3]
scores = df['answer'].apply(lambda truth: map_at_3(truth, top_answers))
scores.mean()

### Question 10: TF-IDF Pipeline Evaluation

In [ ]:
def get_top3_preds(row):
    options = ['A', 'B', 'C', 'D', 'E']
    p_v = vectorizer.transform([row['prompt']])
    s = {opt: cosine_similarity(p_v, vectorizer.transform([row[opt]]))[0][0] for opt in options}
    return sorted(s, key=s.get, reverse=True)

all_scores = df.apply(lambda row: map_at_3(row['answer'], get_top3_preds(row)), axis=1)
all_scores.mean()

In [2]:
import requests

token = "***REMOVED_GITHUB_PAT***"
owner = "23F2004693"
repo = "DL-23F2004693-notebook-t22026"

url = f"https://api.github.com/repos/{owner}/{repo}"

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json"
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)
print(response.json())


Status: 200
{'id': 1277597743, 'node_id': 'R_kgDOTCaYLw', 'name': 'DL-23F2004693-notebook-t22026', 'full_name': '23F2004693/DL-23F2004693-notebook-t22026', 'private': True, 'owner': {'login': '23F2004693', 'id': 188542807, 'node_id': 'U_kgDOCzzvVw', 'avatar_url': 'https://avatars.githubusercontent.com/u/188542807?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/23F2004693', 'html_url': 'https://github.com/23F2004693', 'followers_url': 'https://api.github.com/users/23F2004693/followers', 'following_url': 'https://api.github.com/users/23F2004693/following{/other_user}', 'gists_url': 'https://api.github.com/users/23F2004693/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/23F2004693/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/23F2004693/subscriptions', 'organizations_url': 'https://api.github.com/users/23F2004693/orgs', 'repos_url': 'https://api.github.com/users/23F2004693/repos', 'events_url': 'https://api.github.com/users/23F200

# Milestone 2

Q1-- Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    'csv',
    data_files='//kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
)['train']

def combine_text(row):
    row['combined_text'] = row['prompt'] + ' ' + row['A']
    return row

dataset = dataset.map(combine_text)

text_51 = dataset[51]['combined_text']

print("combined_text:", text_51)
print("\nAnswer - Character length:", len(text_51))

combined_text: Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

Answer - Character length: 614


Q2-- Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer? 

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size: 30522


Q3-- Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token. 

In [5]:
print("[SEP] token ID:", tokenizer.sep_token_id)

[SEP] token ID: 102


Q4 -- Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [7]:
prompts = list(dataset['prompt'])  # Add list() here

encoded = tokenizer(
    prompts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

print("input_ids shape:", encoded['input_ids'].shape)

input_ids shape: torch.Size([2000, 128])


# BERT/RoBERTa Architecture & Attention Mechanisms


Q5-- A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [10]:
768 / 12

64.0

Q6-- Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [11]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

inputs = tokenizer(dataset[0]['prompt'], return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

print("last_hidden_state shape:", outputs.last_hidden_state.shape)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


last_hidden_state shape: torch.Size([1, 31, 768])


Q7--   Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  
*


In [12]:
cls_vector = outputs.last_hidden_state[0][0]
print("Sum:", round(cls_vector[:5].sum().item(), 4))

Sum: -1.2001


Q8-- Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

*


In [13]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs = tokenizer(text, return_tensors='pt')

tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print("Tokens:", tokens)
fusion_idx = tokens.index('fusion')

with torch.no_grad():
    out = model_attn(**inputs)

weight = out.attentions[-1][0][0][0][fusion_idx]
print("Attention weight:", round(weight.item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Attention weight: 0.1025


# Context-Aware Embeddings 
Q9-- Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.


In [14]:
from sentence_transformers import SentenceTransformer, util

minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

emb_prompt = minilm.encode(dataset[0]['prompt'])
emb_b      = minilm.encode(dataset[0]['B'])

print("Cosine similarity:", round(util.cos_sim(emb_prompt, emb_b).item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine similarity: 0.7658


Q10-- Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

options = ['A', 'B', 'C', 'D', 'E']

def get_top3_tfidf(row):
    docs = [row['prompt']] + [row[o] for o in options]
    tfidf = TfidfVectorizer().fit_transform(docs)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:])[0]
    return [options[i] for i in np.argsort(sims)[::-1][:3]]

def get_top3_minilm(row):
    p = minilm.encode(row['prompt'])
    o = minilm.encode([row[opt] for opt in options])
    sims = util.cos_sim(p, o)[0].numpy()
    return [options[i] for i in np.argsort(sims)[::-1][:3]]

def ap3(top3, correct):
    for i, p in enumerate(top3):
        if p == correct:
            return 1 / (i + 1)
    return 0

tfidf_preds, minilm_preds = [], []
for row in dataset:
    tfidf_preds.append(get_top3_tfidf(row))
    minilm_preds.append(get_top3_minilm(row))

map3 = np.mean([ap3(minilm_preds[i], dataset[i]['answer']) for i in range(len(dataset))])
print("MiniLM MAP@3:", round(map3, 4))

count = sum(
    1 for i in range(len(dataset))
    if dataset[i]['answer'] not in tfidf_preds[i]
    and dataset[i]['answer'] in minilm_preds[i]
)
print("TF-IDF miss but MiniLM hit:", count)

MiniLM MAP@3: 0.4231
TF-IDF miss but MiniLM hit: 564


# Zero-shot classification concepts 
Q11-- Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [16]:
from transformers import pipeline

zsc = pipeline("zero-shot-classification")

row1 = dataset[1]
labels = [row1['A'], row1['B'], row1['C']]

result = zsc(row1['prompt'], candidate_labels=labels)
print("Top score:", round(result['scores'][0], 4))
print("All scores:", result['scores'])

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Top score: 0.4575
All scores: [0.4574527442455292, 0.2750643789768219, 0.26748284697532654]


Q12-- Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [17]:
result_multi = zsc(row1['prompt'], candidate_labels=labels, multi_label=True)

diff = abs(sum(result['scores']) - sum(result_multi['scores']))
print("Absolute difference:", round(diff, 4))

Absolute difference: 0.9995


Q13-- Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [19]:
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

t5_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")

row0 = dataset[0]
input_text = f"Question: {row0['prompt']}. Is the correct answer A: {row0['A']} or B: {row0['B']}? Answer with just the letter A or B."

inputs = t5_tokenizer(input_text, return_tensors="pt")

with torch.no_grad():
    output_ids = t5_model.generate(**inputs, max_new_tokens=5)

result = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print("Output:", result)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Output: B
